# 02 Model A: CYP3A4 inhibition

Train the model that answers condition 1 of the screen: does this herb compound inhibit CYP3A4? It learns from the roughly 12k compound `CYP3A4_Veith` set loaded in notebook 01.

The target is ROC-AUC >= 0.80 on held out data. We also fit the applicability domain here, because Model A is the leg that gets pointed at plant natural products, which look quite unlike the synthetic compounds it was trained on.

Two splits are reported, not one. A random split lets near duplicate analogues sit on both sides and inflates the AUC; a scaffold split puts whole chemical series on one side only, which is closer to what happens when the model meets a plant compound it has never seen a relative of. **The scaffold number is the honest one to quote.**

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import features
import models
import applicability

PROCESSED_DIR = os.path.join("..", "data", "processed")

## Load the matrices from notebook 01

In [ ]:
X = np.load(os.path.join(PROCESSED_DIR, "X_cyp3a4.npy"))
y = np.load(os.path.join(PROCESSED_DIR, "y_cyp3a4.npy"))
cyp_df = pd.read_csv(os.path.join(PROCESSED_DIR, "cyp3a4_clean.csv"))

print("X:", X.shape)
print("y:", y.shape, "positive fraction:", round(float(y.mean()), 3))
print("smiles rows:", len(cyp_df))
assert X.shape[0] == y.shape[0] == len(cyp_df), "notebook 01 outputs are out of sync"

## Random split`stratify=y` keeps the label balance the same on both sides of the split, whichmatters because the set is uneven and a lopsided test set would make the AUCdepend on the split rather than on the model.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, stratify=y, random_state=42
)

print("train:", X_train.shape, "test:", X_test.shape)
print("train positive fraction:", round(float(y_train.mean()), 3))
print("test  positive fraction:", round(float(y_test.mean()), 3))

## Train Model A

In [ ]:
model_a = models.train_rf(X_train, y_train)
print("trained", model_a.n_estimators, "trees on", X_train.shape[0], "compounds")

## Evaluate on the random split

In [ ]:
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
)

probs = model_a.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

auc = roc_auc_score(y_test, probs)
print(f"ROC-AUC: {auc:.4f}   (target >= 0.80 -> {'MET' if auc >= 0.80 else 'NOT MET'})")
print(f"average precision: {average_precision_score(y_test, probs):.4f}")
print()
print("confusion matrix at threshold 0.5")
print("                 pred 0   pred 1")
cm = confusion_matrix(y_test, preds)
print(f"  actual 0    {cm[0,0]:8d} {cm[0,1]:8d}")
print(f"  actual 1    {cm[1,0]:8d} {cm[1,1]:8d}")
print()
print(classification_report(y_test, preds, target_names=["non-inhibitor", "inhibitor"]))

The screen cares most about **false positives**: a wrongly flagged pair sendssomeone to the lab to measure nothing. That is the precision number on theinhibitor row, so it is worth quoting alongside the AUC rather than hidingbehind the single headline figure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

fpr, tpr, _ = roc_curve(y_test, probs)
axes[0].plot(fpr, tpr, label=f"Model A (AUC = {auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="chance")
axes[0].set_xlabel("false positive rate")
axes[0].set_ylabel("true positive rate")
axes[0].set_title("Model A ROC, random split")
axes[0].legend()

precision, recall, _ = precision_recall_curve(y_test, probs)
axes[1].plot(recall, precision)
axes[1].axhline(y_test.mean(), color="k", linestyle="--", linewidth=0.8,
                label=f"baseline = {y_test.mean():.2f}")
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Model A precision-recall")
axes[1].legend()

plt.tight_layout()
plt.show()

## Scaffold splitGroup the compounds by Bemis-Murcko scaffold and put whole scaffold groups onone side of the split only. A large drop against the random split means themodel was recognising close analogues rather than learning chemistry, which isexactly the failure mode that matters when it is later pointed at plantcompounds with scaffolds the training set never contained.

In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def scaffold_split_indices(smiles_list, test_fraction=0.2, seed=42):
    # Group row indices by Murcko scaffold, then fill the test set with whole
    # groups, largest first, until it is big enough. Assigning groups rather
    # than molecules is the whole point: no scaffold may appear on both sides.
    groups = defaultdict(list)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            groups[f"__unparsed_{i}"].append(i)
            continue
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        groups[scaffold].append(i)

    ordered = sorted(groups.values(), key=len, reverse=True)
    n_test_target = int(len(smiles_list) * test_fraction)
    test_idx, train_idx = [], []
    for group in ordered:
        if len(test_idx) < n_test_target:
            test_idx.extend(group)
        else:
            train_idx.extend(group)
    return np.array(train_idx), np.array(test_idx), len(groups)

smiles = cyp_df["Drug"].tolist()
sc_train, sc_test, n_scaffolds = scaffold_split_indices(smiles)
print(f"{n_scaffolds} distinct scaffolds over {len(smiles)} compounds")
print("scaffold train:", sc_train.shape, "scaffold test:", sc_test.shape)

In [ ]:
model_a_sc = models.train_rf(X[sc_train], y[sc_train])
sc_probs = model_a_sc.predict_proba(X[sc_test])[:, 1]
sc_auc = roc_auc_score(y[sc_test], sc_probs)

print(f"random split   ROC-AUC: {auc:.4f}")
print(f"scaffold split ROC-AUC: {sc_auc:.4f}")
print(f"drop: {auc - sc_auc:.4f}")
print()
if sc_auc >= 0.80:
    print("scaffold AUC meets the 0.80 target")
else:
    print("scaffold AUC BELOW the 0.80 target - report this, do not quote the random number alone")

## Applicability domainFitted on the **training** fingerprints only. Fitting it on everything wouldleak the test set into the gate, and fitting it after seeing the herb resultswould turn it into a knob for getting a nicer answer.The floor is the 5th percentile of training density, so by construction about5 percent of the training set itself falls below it. That is the intendedcalibration: the gate is as permissive as the sparsest corner of the data themodel was actually fitted on.

In [ ]:
domain = applicability.fit_domain(X_train)

print(f"similarity floor: {domain['floor']:.4f}")
print(f"k neighbours:     {domain['k']}")
print(f"training density: mean {domain['train_densities'].mean():.4f}, "
      f"min {domain['train_densities'].min():.4f}, "
      f"max {domain['train_densities'].max():.4f}")

held_out_scores = applicability.density_scores(X_test, domain)
outside = (held_out_scores < domain["floor"]).mean()
print(f"\nheld out compounds falling outside the domain: {outside:.1%}")
print("(a sanity check: these are the same kind of synthetic compounds as the")
print(" training set, so this should be small. Herb compounds will be far higher.)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(domain["train_densities"], bins=50, alpha=0.7, label="training set")
ax.hist(held_out_scores, bins=50, alpha=0.7, label="held out")
ax.axvline(domain["floor"], color="red", linestyle="--",
           label=f"floor = {domain['floor']:.3f}")
ax.set_xlabel(f"mean Tanimoto to {domain['k']} nearest training neighbours")
ax.set_ylabel("compounds")
ax.set_title("Applicability domain density")
ax.legend()
plt.tight_layout()
plt.show()

## Save for notebooks 04 and 06 and the demo app

In [ ]:
joblib.dump(model_a, os.path.join(PROCESSED_DIR, "model_a.joblib"))
applicability.save_domain(domain, os.path.join(PROCESSED_DIR, "domain_a.npz"))

# Keep the split indices so notebook 04 explains the model on data it was not
# fitted on, rather than on its own training rows.
np.savez(
    os.path.join(PROCESSED_DIR, "split_a.npz"),
    idx_train=idx_train, idx_test=idx_test,
    scaffold_train=sc_train, scaffold_test=sc_test,
)

metrics = {
    "model": "A (CYP3A4_Veith)",
    "n_compounds": int(X.shape[0]),
    "positive_fraction": float(y.mean()),
    "roc_auc_random": float(auc),
    "roc_auc_scaffold": float(sc_auc),
    "domain_floor": float(domain["floor"]),
}
pd.Series(metrics).to_csv(os.path.join(PROCESSED_DIR, "metrics_model_a.csv"))
print("saved model, domain, splits and metrics to", PROCESSED_DIR)
print()
for k, v in metrics.items():
    print(f"  {k}: {v}")